# 01 Malaysia Data Preparation

Validate the retained model-ready state panel and generated dashboard prediction artifact used by the public repository.


This notebook checks the stable downstream files after Notebook 00 has established the raw DOSM poverty-label lineage. It does not recreate satellite/geospatial features; it validates the retained model-ready panel and the generated selected-model prediction artifact.


In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

QA_DIR = PROJECT_ROOT / "outputs" / "qa"
REPORTING_DIR = PROJECT_ROOT / "outputs" / "reporting"
INTERMEDIATE_DIR = PROJECT_ROOT / "outputs" / "intermediate"
for path in [QA_DIR, REPORTING_DIR, INTERMEDIATE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def write_json(path: Path, payload: dict) -> dict:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    return payload

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

In [2]:
import yaml

cfg = yaml.safe_load((PROJECT_ROOT / "config" / "malaysia.yaml").read_text(encoding="utf-8"))
paths = {key: PROJECT_ROOT / value for key, value in cfg.items() if isinstance(value, str) and Path(value).suffix}

state_panel = pd.read_csv(paths["state_panel"])
dashboard_predictions = pd.read_csv(paths["model_predictions"])
clean_dosm_labels = pd.read_csv(paths["clean_dosm_poverty"])

required_targets = ["poverty_absolute", "poverty_relative", "poverty_hardcore"]
required_features = [
    "wmean_ntl_mean", "wmean_evi_mean", "wmean_ndvi_mean",
    "wmean_frac_urban", "pop_sum_grid", "wmean_elevation",
]
missing_targets = [col for col in required_targets if col not in state_panel.columns]
missing_features = [col for col in required_features if col not in state_panel.columns]

summary = {
    "country": "Malaysia",
    "active_scope": "state_level_only",
    "state_panel_rows": int(len(state_panel)),
    "state_panel_columns": int(len(state_panel.columns)),
    "state_count": int(state_panel["state_key"].nunique()) if "state_key" in state_panel else None,
    "survey_years": sorted(int(year) for year in state_panel["year"].dropna().unique()) if "year" in state_panel else [],
    "clean_dosm_label_rows": int(len(clean_dosm_labels)),
    "dashboard_prediction_artifact_rows": int(len(dashboard_predictions)),
    "missing_required_targets": missing_targets,
    "missing_required_features": missing_features,
    "output_contract": "Notebook 00 proves DOSM label lineage; this notebook validates stable downstream state-level artifacts.",
}
write_json(QA_DIR / "malaysia_data_preparation_summary.json", summary)
pd.DataFrame([summary])

,country,active_scope,state_panel_rows,state_panel_columns,state_count,survey_years,clean_dosm_label_rows,dashboard_prediction_artifact_rows,missing_required_targets,missing_required_features,output_contract
0,Malaysia,state_level_only,158,70,16,"[2002, 2004, 2007, 2009, 2012, 2014, 2016, 201...",158,158,[],[],Notebook 00 proves DOSM label lineage; this no...
